In [1]:
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

In [2]:
# Set image size and parameters
img_size = (224, 224)
batch_size = 32
num_classes = 7

In [5]:
# Step 1: Data augmentation and preprocessing for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

valid_test_datagen = ImageDataGenerator(rescale=1./255)

# Load train and test data
train_generator = train_datagen.flow_from_directory(
    'Skin_Cancer_Dataset/train_dir',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = valid_test_datagen.flow_from_directory(
    'Skin_Cancer_Dataset/val_dir',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

Found 38569 images belonging to 7 classes.
Found 938 images belonging to 7 classes.


In [7]:
# Step 2: Load your saved MobileNet and VGG models
mobilenet = load_model('melanoma_skin_cancer_MobileNet_model.h5')
vgg = load_model('melanoma_skin_cancer_VGG_model.h5')

In [8]:
# Step 3: Remove the top layers (the classification head) to get the features
def remove_top(model):
    model.layers[-1].activation = None
    return models.Model(inputs=model.input, outputs=model.layers[-2].output)

mobilenet_features = remove_top(mobilenet)
vgg_features = remove_top(vgg)

# Step 4: Create a shared input layer
input_layer = layers.Input(shape=(224, 224, 3))

# Pass the same input through both models
mobilenet_output = mobilenet_features(input_layer)
vgg_output = vgg_features(input_layer)

In [9]:
# Step 5: Combine the feature outputs from MobileNet and VGG
combined = layers.concatenate([mobilenet_output, vgg_output])

# Step 6: Add new fully connected layers for classification
x = layers.Dense(512, activation='relu')(combined)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.5)(x)
predictions = layers.Dense(num_classes, activation='softmax')(x)

In [ ]:
# Step 7: Define the new model
ensemble_model = models.Model(inputs=input_layer, outputs=predictions)

# Step 8: Compile the model
ensemble_model.compile(optimizer=Adam(learning_rate=1e-4),
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

# Step 9: Train the model
history = ensemble_model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // batch_size,
    validation_data=test_generator,
    validation_steps=test_generator.samples // batch_size,
    epochs=50
)

# Step 10: Evaluate the model
test_loss, test_acc = ensemble_model.evaluate(test_generator, steps=test_generator.samples // batch_size)

In [14]:
print(f'Test accuracy: {test_acc}')

# Optionally, save the trained model
ensemble_model.save('Skin_Cancer_model.keras')

Test accuracy: 0.8868534564971924
